## Example 1: Basic DataFrame operations and compatibility

In [ ]:
# Example 1: Basic DataFrame operations and compatibility
import narwhals as nw
import pandas as pd
import numpy as np

print("Example 1: Basic DataFrame Operations and Compatibility")

try:
    print(f"narwhals version: {getattr(nw, '__version__', 'unknown')}")
    
    # Create sample data with different types
    sample_data = {
        'id': [1, 2, 3, 4, 5],
        'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
        'age': [25, 30, 35, 28, 32],
        'salary': [50000.0, 60000.0, 70000.0, 55000.0, 65000.0],
        'department': ['Engineering', 'Sales', 'Engineering', 'Marketing', 'Sales'],
        'active': [True, False, True, True, False]
    }
    
    print(f"Creating DataFrames with sample employee data:")
    print(f"Records: {len(sample_data['id'])}")
    print(f"Columns: {list(sample_data.keys())}")
    
    # Create pandas DataFrame
    df_pandas = pd.DataFrame(sample_data)
    print(f"\nPandas DataFrame:")
    print(df_pandas.head())
    print(f"Pandas dtypes: {df_pandas.dtypes.to_dict()}")
    
    # Convert to narwhals DataFrame
    df_nw = nw.from_native(df_pandas)
    print(f"\nNarwhals DataFrame:")
    print(f"Type: {type(df_nw)}")
    print(f"Shape: {df_nw.shape}")
    print(f"Columns: {df_nw.columns}")
    
    # Basic operations - selection
    print(f"\nColumn selection operations:")
    
    # Select specific columns
    selected = df_nw.select('name', 'age', 'salary')
    print(f"Selected columns (name, age, salary):")
    print(selected.to_pandas())
    
    # Select with column expressions
    salary_info = df_nw.select(
        nw.col('name'),
        nw.col('salary'),
        (nw.col('salary') / 1000).alias('salary_k')
    )
    print(f"\nSalary with calculated column (in thousands):")
    print(salary_info.to_pandas())
    
    # Filtering operations
    print(f"\nFiltering operations:")
    
    # Simple filter
    high_earners = df_nw.filter(nw.col('salary') > 60000)
    print(f"High earners (salary > 60k):")
    print(high_earners.to_pandas())
    
    # Multiple conditions
    engineering_active = df_nw.filter(
        (nw.col('department') == 'Engineering') & (nw.col('active') == True)
    )
    print(f"\nActive Engineering employees:")
    print(engineering_active.to_pandas())
    
    # Age-based filtering
    experienced = df_nw.filter(nw.col('age') >= 30)
    print(f"\nExperienced employees (age >= 30):")
    print(experienced.select('name', 'age', 'department').to_pandas())
    
    # Aggregation operations
    print(f"\nAggregation operations:")
    
    # Basic aggregations
    summary = df_nw.select([
        nw.col('salary').mean().alias('avg_salary'),
        nw.col('salary').min().alias('min_salary'),
        nw.col('salary').max().alias('max_salary'),
        nw.col('age').mean().alias('avg_age'),
        nw.len().alias('total_count')
    ])
    print(f"Summary statistics:")
    print(summary.to_pandas())
    
    # Group by operations
    dept_summary = df_nw.group_by('department').agg([
        nw.col('salary').mean().alias('avg_salary'),
        nw.col('age').mean().alias('avg_age'),
        nw.len().alias('count')
    ])
    print(f"\nDepartment summary:")
    print(dept_summary.to_pandas())
    
    # Sorting operations
    print(f"\nSorting operations:")
    
    # Sort by single column
    by_salary = df_nw.sort('salary', descending=True)
    print(f"Sorted by salary (descending):")
    print(by_salary.select('name', 'salary', 'department').to_pandas())
    
    # Sort by multiple columns
    by_dept_age = df_nw.sort(['department', 'age'])
    print(f"\nSorted by department, then age:")
    print(by_dept_age.select('name', 'department', 'age').to_pandas())
    
    # Demonstrate backend independence
    print(f"\nBackend independence demonstration:")
    
    # Get native backend info
    native_namespace = nw.get_native_namespace(df_pandas)
    print(f"Native namespace: {native_namespace}")
    
    # Convert back to native format
    result_native = selected.to_native()
    print(f"Native result type: {type(result_native)}")
    print(f"Is pandas DataFrame: {isinstance(result_native, pd.DataFrame)}")
    
    # Show compatibility features
    print(f"\nCompatibility features:")
    compatibility_features = [
        "Universal API: Same syntax works across pandas, Polars, PyArrow",
        "Type preservation: Maintains column types during operations", 
        "Lazy evaluation: Supports both eager and lazy execution modes",
        "Memory efficiency: Optimized operations for large datasets",
        "Expression system: Powerful column expression language",
        "Backend agnostic: Write once, run on multiple DataFrame libraries"
    ]
    
    for feature in compatibility_features:
        print(f"  • {feature}")
    
    print("Basic DataFrame operations completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating DataFrame compatibility concepts:")
    concepts = [
        "Universal DataFrame API: Consistent interface across different backends",
        "Column selection: Select and transform columns with unified syntax",
        "Filtering operations: Apply conditions using expression language",
        "Aggregations: Compute statistics with backend-agnostic functions",
        "Backend independence: Switch between pandas/Polars without code changes"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise

## Example 2: Lazy evaluation and query optimization

In [ ]:
# Example 2: Lazy evaluation and query optimization
import narwhals as nw
import pandas as pd
import time

print("Example 2: Lazy Evaluation and Query Optimization")

try:
    # Create larger dataset for performance demonstration
    np.random.seed(42)
    n_rows = 10000
    
    large_data = {
        'user_id': range(n_rows),
        'transaction_amount': np.random.uniform(10, 1000, n_rows),
        'category': np.random.choice(['Food', 'Shopping', 'Transport', 'Entertainment', 'Utilities'], n_rows),
        'date': pd.date_range('2023-01-01', periods=n_rows, freq='h'),  # pandas 3 removed the uppercase 'H' alias
        'is_weekend': np.random.choice([True, False], n_rows, p=[0.3, 0.7]),
        'region': np.random.choice(['North', 'South', 'East', 'West'], n_rows)
    }
    
    print(f"Created dataset with {n_rows:,} transactions")
    print(f"Columns: {list(large_data.keys())}")
    
    # Create DataFrames
    df_pandas = pd.DataFrame(large_data)
    df_nw = nw.from_native(df_pandas)
    
    print(f"DataFrame shape: {df_nw.shape}")
    print(f"Memory usage: ~{df_pandas.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    # Demonstrate lazy evaluation
    print(f"\nLazy evaluation demonstration:")
    
    # Create lazy frame
    lazy_df = df_nw.lazy()
    print(f"Lazy frame type: {type(lazy_df)}")
    
    # Chain multiple operations (not executed yet)
    lazy_query = (
        lazy_df
        .filter(nw.col('transaction_amount') > 100)
        .filter(nw.col('category') == 'Shopping')
        .with_columns([
            (nw.col('transaction_amount') * 0.1).alias('tax'),
            nw.col('date').dt.hour().alias('hour')
        ])
        .group_by(['region', 'is_weekend'])
        .agg([
            nw.col('transaction_amount').mean().alias('avg_amount'),
            nw.col('transaction_amount').sum().alias('total_amount'),
            nw.len().alias('transaction_count')
        ])
        .sort('total_amount', descending=True)
    )
    
    print(f"Lazy query created (not executed)")
    print(f"Query type: {type(lazy_query)}")
    
    # Execute lazy query
    print(f"\nExecuting lazy query...")
    start_time = time.time()
    result = lazy_query.collect()
    lazy_time = time.time() - start_time
    
    print(f"Lazy execution time: {lazy_time:.4f}s")
    print(f"Result shape: {result.shape}")
    print(f"Shopping transactions > $100 by region and weekend:")
    print(result.to_pandas())
    
    # Compare with eager evaluation
    print(f"\nComparing with eager evaluation:")
    
    start_time = time.time()
    eager_result = (
        df_nw
        .filter(nw.col('transaction_amount') > 100)
        .filter(nw.col('category') == 'Shopping')
        .with_columns([
            (nw.col('transaction_amount') * 0.1).alias('tax'),
            nw.col('date').dt.hour().alias('hour')
        ])
        .group_by(['region', 'is_weekend'])
        .agg([
            nw.col('transaction_amount').mean().alias('avg_amount'),
            nw.col('transaction_amount').sum().alias('total_amount'),
            nw.len().alias('transaction_count')
        ])
        .sort('total_amount', descending=True)
    )
    eager_time = time.time() - start_time
    
    print(f"Eager execution time: {eager_time:.4f}s")
    print(f"Performance comparison: {abs(eager_time - lazy_time):.4f}s difference")
    
    # Complex analytical query with lazy evaluation
    print(f"\nComplex analytical query:")
    
    analytical_query = (
        df_nw.lazy()
        .with_columns([
            nw.col('date').dt.month().alias('month'),
            nw.col('date').dt.weekday().alias('day_of_week'),  # narwhals 2.x: `dt.weekday()` (no `dt.day_of_week`)
            (nw.col('transaction_amount') > nw.col('transaction_amount').median()).alias('high_value')
        ])
        .filter(nw.col('transaction_amount') > 50)
        .group_by(['category', 'month'])
        .agg([
            nw.col('transaction_amount').mean().alias('avg_amount'),
            nw.col('transaction_amount').std().alias('std_amount'),
            nw.col('high_value').sum().alias('high_value_count'),
            nw.len().alias('total_transactions')
        ])
        .with_columns([
            (nw.col('high_value_count') / nw.col('total_transactions')).alias('high_value_ratio')
        ])
        .filter(nw.col('total_transactions') > 10)
        .sort(['category', 'month'])
    )
    
    analytical_result = analytical_query.collect()
    print(f"Monthly category analysis (transactions > $50):")
    print(analytical_result.head(10).to_pandas())
    
    # Demonstrate query optimization benefits
    print(f"\nQuery optimization benefits:")
    
    # Predicate pushdown simulation
    filtered_first = (
        df_nw.lazy()
        .filter(nw.col('region') == 'North')  # Filter early
        .filter(nw.col('transaction_amount') > 500)  # Additional filter
        .select(['user_id', 'transaction_amount', 'category'])  # Select only needed columns
        .group_by('category')
        .agg([
            nw.col('transaction_amount').mean().alias('avg_amount'),
            nw.len().alias('count')
        ])
        .collect()
    )
    
    print(f"Optimized query (filter + select early):")
    print(filtered_first.to_pandas())
    
    # Window functions and advanced operations
    print(f"\nWindow functions and advanced operations:")
    
    try:
        window_query = (
            df_nw.lazy()
            .filter(nw.col('category') == 'Food')
            .with_columns([
                nw.col('transaction_amount').rank().over('region').alias('amount_rank'),
                nw.col('transaction_amount').mean().over('region').alias('region_avg')
            ])
            .with_columns([
                (nw.col('transaction_amount') - nw.col('region_avg')).alias('amount_diff')
            ])
            .filter(nw.col('amount_rank') <= 5)  # Top 5 per region
            .select(['user_id', 'region', 'transaction_amount', 'amount_rank', 'amount_diff'])
            .sort(['region', 'amount_rank'])
            .collect()
        )
        
        print(f"Top 5 food transactions per region with ranking:")
        print(window_query.head(15).to_pandas())
        
    except Exception as e:
        print(f"Window functions not available: {e}")
        print("Simulating with group operations...")
        
        # Alternative approach without window functions
        region_stats = (
            df_nw
            .filter(nw.col('category') == 'Food')
            .group_by('region')
            .agg([
                nw.col('transaction_amount').mean().alias('region_avg')
            ])
        )
        print(f"Region averages for food category:")
        print(region_stats.to_pandas())
    
    # Show lazy evaluation benefits
    print(f"\nLazy evaluation benefits:")
    lazy_benefits = [
        "Query optimization: Automatically optimize query execution plans",
        "Memory efficiency: Process data without loading entire datasets",
        "Predicate pushdown: Apply filters early to reduce data volume",
        "Column pruning: Only load necessary columns from storage",
        "Parallel execution: Leverage multiple cores for computation",
        "Deferred execution: Build complex queries before execution"
    ]
    
    for benefit in lazy_benefits:
        print(f"  • {benefit}")
    
    print("Lazy evaluation and query optimization completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating lazy evaluation concepts:")
    concepts = [
        "Lazy evaluation: Defer computation until results are needed",
        "Query optimization: Automatic optimization of execution plans",
        "Predicate pushdown: Apply filters early in the pipeline",
        "Column selection: Load only required columns for efficiency",
        "Complex analytics: Chain multiple operations efficiently"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise

## Example 3: Multi-backend compatibility and DataFrame conversion

In [ ]:
# Example 3: Multi-backend compatibility and DataFrame conversion
import narwhals as nw
import pandas as pd
import numpy as np

print("Example 3: Multi-backend Compatibility and DataFrame Conversion")

try:
    # Create test data for conversion demonstrations
    test_data = {
        'product_id': ['P001', 'P002', 'P003', 'P004', 'P005'],
        'price': [19.99, 29.99, 39.99, 49.99, 59.99],
        'category': ['Electronics', 'Clothing', 'Electronics', 'Home', 'Clothing'],
        'stock': [100, 50, 75, 25, 125],
        'rating': [4.5, 3.8, 4.2, 4.7, 3.9]
    }
    
    print(f"Multi-backend compatibility demonstration:")
    print(f"Test data: {len(test_data['product_id'])} products")
    
    # Pandas DataFrame
    df_pandas = pd.DataFrame(test_data)
    print(f"\n1. Pandas DataFrame:")
    print(f"   Type: {type(df_pandas)}")
    print(f"   Shape: {df_pandas.shape}")
    
    # Convert pandas to narwhals
    df_nw_from_pandas = nw.from_native(df_pandas)
    print(f"\n2. Narwhals from Pandas:")
    print(f"   Type: {type(df_nw_from_pandas)}")
    print(f"   Backend: {nw.get_native_namespace(df_pandas)}")
    
    # Perform operations through narwhals
    processed = (
        df_nw_from_pandas
        .with_columns([
            (nw.col('price') * 1.2).alias('price_with_tax'),
            (nw.col('rating') * 20).alias('rating_percent')
        ])
        .filter(nw.col('stock') > 30)
        .select(['product_id', 'category', 'price', 'price_with_tax', 'rating_percent'])
    )
    
    print(f"\n3. Processed through narwhals:")
    print(processed.to_pandas())
    
    # Demonstrate from_dict functionality
    dict_data = {
        'name': ['Alice', 'Bob', 'Charlie'],
        'score': [95, 87, 92],
        'grade': ['A', 'B', 'A']
    }
    
    # narwhals 2.x: plain Python lists carry no backend, so `from_dict` must be told which one to build.
    df_from_dict = nw.from_dict(dict_data, backend="pandas")
    print(f"\n4. Created from dictionary:")
    print(f"   Type: {type(df_from_dict)}")
    print(df_from_dict.to_pandas())
    
    # Schema validation and type checking
    print(f"\n5. Schema validation:")
    
    schema = df_nw_from_pandas.schema
    print(f"   Schema: {schema}")
    
    for col_name, dtype in schema.items():
        print(f"   {col_name}: {dtype}")
    
    # Demonstrate type preservation across operations
    print(f"\n6. Type preservation:")
    
    type_demo = df_nw_from_pandas.select([
        nw.col('product_id').cast(nw.String).alias('id_string'),
        nw.col('price').cast(nw.Float64).alias('price_float64'),
        nw.col('stock').cast(nw.Int32).alias('stock_int32')
    ])
    
    print(f"   Cast operations schema: {type_demo.schema}")
    print(type_demo.to_pandas().dtypes)
    
    # Cross-backend function compatibility
    print(f"\n7. Cross-backend function compatibility:")
    
    # String operations
    string_ops = df_nw_from_pandas.with_columns([
        nw.col('product_id').str.len_chars().alias('id_length'),  # narwhals 2.x: `str.len_chars()`
        nw.col('category').str.to_uppercase().alias('category_upper')
    ]).select(['product_id', 'id_length', 'category', 'category_upper'])
    
    print(f"   String operations:")
    print(string_ops.to_pandas())
    
    # Mathematical operations
    math_ops = df_nw_from_pandas.with_columns([
        nw.col('price').round(0).alias('price_rounded'),
        (nw.col('rating') * nw.col('stock')).alias('weighted_score'),
        nw.col('stock').sqrt().alias('stock_sqrt')
    ]).select(['product_id', 'price', 'price_rounded', 'weighted_score', 'stock_sqrt'])
    
    print(f"\n   Mathematical operations:")
    print(math_ops.to_pandas())
    
    # Conditional operations  
    conditional_ops = df_nw_from_pandas.with_columns([
        nw.when(nw.col('price') > 40)
        .then(nw.lit('Premium'))
        .when(nw.col('price') > 25) 
        .then(nw.lit('Standard'))
        .otherwise(nw.lit('Budget'))
        .alias('price_tier'),
        
        nw.when(nw.col('stock') < 50)
        .then(nw.lit('Low'))
        .otherwise(nw.lit('Sufficient'))
        .alias('stock_status')
    ])
    
    print(f"\n   Conditional operations:")
    print(conditional_ops.select(['product_id', 'price', 'price_tier', 'stock', 'stock_status']).to_pandas())
    
    # Aggregation compatibility
    print(f"\n8. Aggregation compatibility:")
    
    category_agg = df_nw_from_pandas.group_by('category').agg([
        nw.col('price').mean().alias('avg_price'),
        nw.col('price').min().alias('min_price'),  
        nw.col('price').max().alias('max_price'),
        nw.col('stock').sum().alias('total_stock'),
        nw.col('rating').mean().alias('avg_rating'),
        nw.len().alias('count')
    ])
    
    print(f"   Category aggregations:")
    print(category_agg.to_pandas())
    
    # Performance comparison across backends
    print(f"\n9. Backend detection and capabilities:")
    
    # Check backend capabilities
    native_ns = nw.get_native_namespace(df_pandas)
    print(f"   Native namespace: {native_ns}")
    print(f"   Has lazy evaluation: {hasattr(df_nw_from_pandas, 'lazy')}")
    print(f"   Supports collect: {hasattr(df_nw_from_pandas.lazy() if hasattr(df_nw_from_pandas, 'lazy') else None, 'collect')}")
    
    # Conversion back to native format
    native_result = processed.to_native()
    print(f"\n10. Native conversion:")
    print(f"    Result type: {type(native_result)}")
    print(f"    Is pandas: {isinstance(native_result, pd.DataFrame)}")
    
    # Show compatibility benefits
    print(f"\nCompatibility benefits:")
    compatibility_benefits = [
        "Write once, run everywhere: Same code works across DataFrame libraries",
        "Type preservation: Maintain data types during backend conversions",
        "Schema validation: Consistent schema handling across backends",
        "Function compatibility: Universal function interface for all backends",
        "Performance optimization: Leverage each backend's strengths",
        "Migration support: Easy switching between DataFrame libraries"
    ]
    
    for benefit in compatibility_benefits:
        print(f"  • {benefit}")
    
    print("Multi-backend compatibility completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating multi-backend concepts:")
    concepts = [
        "Backend abstraction: Universal interface across DataFrame libraries",
        "Type preservation: Consistent data types during conversions",
        "Schema validation: Unified schema handling and validation",
        "Function compatibility: Same functions work across all backends",
        "Native conversion: Convert back to original library format"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise

## Example 4: Advanced expressions and data transformations

In [ ]:
# Example 4: Advanced expressions and data transformations
import narwhals as nw
import pandas as pd

print("Example 4: Advanced Expressions and Data Transformations")

try:
    # Create complex dataset for advanced operations
    sales_data = {
        'order_id': ['ORD001', 'ORD002', 'ORD003', 'ORD004', 'ORD005', 'ORD006'],
        'customer': ['Alice', 'Bob', 'Charlie', 'Alice', 'Diana', 'Bob'],
        'amount': [150.0, 200.0, 75.0, 300.0, 125.0, 180.0],
        'discount': [0.1, 0.0, 0.15, 0.05, 0.2, 0.1],
        'quantity': [3, 2, 1, 4, 2, 3],
        'category': ['Electronics', 'Clothing', 'Home', 'Electronics', 'Clothing', 'Home']
    }
    
    df = nw.from_native(pd.DataFrame(sales_data))
    print(f"Sales data with {df.shape[0]} orders")
    print(df.to_pandas())
    
    # Complex column expressions
    print(f"\nAdvanced column expressions:")
    
    enhanced = df.with_columns([
        (nw.col('amount') * (1 - nw.col('discount'))).alias('final_amount'),
        (nw.col('amount') / nw.col('quantity')).alias('price_per_unit'),
        nw.col('customer').str.len_chars().alias('customer_name_length'),  # narwhals 2.x: `str.len_chars()`
        nw.when(nw.col('discount') > 0.1)
        .then(nw.lit('High Discount'))
        .when(nw.col('discount') > 0.05)
        .then(nw.lit('Medium Discount'))  
        .otherwise(nw.lit('Low/No Discount'))
        .alias('discount_tier')
    ])
    
    print(enhanced.select(['order_id', 'amount', 'discount', 'final_amount', 'discount_tier']).to_pandas())
    
    # String manipulations
    print(f"\nString manipulations:")
    
    string_ops = df.with_columns([
        nw.col('customer').str.to_uppercase().alias('customer_upper'),
        nw.col('order_id').str.slice(3, None).alias('order_number'),
        nw.concat_str([nw.col('customer'), nw.lit(' - '), nw.col('category')]).alias('customer_category')
    ])
    
    print(string_ops.select(['customer', 'customer_upper', 'order_id', 'order_number', 'customer_category']).to_pandas())
    
    # Mathematical expressions
    print(f"\nMathematical expressions:")
    
    math_ops = df.with_columns([
        nw.col('amount').round(0).alias('amount_rounded'),
        (nw.col('amount') * nw.col('quantity')).sqrt().alias('amount_qty_sqrt'),
        nw.col('discount').abs().alias('discount_abs'),
        (nw.col('amount').max() - nw.col('amount')).alias('amount_from_max')
    ])
    
    print(math_ops.select(['order_id', 'amount', 'amount_rounded', 'amount_qty_sqrt', 'amount_from_max']).to_pandas())
    
    # Conditional logic and case statements
    print(f"\nConditional logic:")
    
    conditional = df.with_columns([
        nw.when(nw.col('amount') >= 200)
        .then(nw.lit('Large Order'))
        .when(nw.col('amount') >= 100)
        .then(nw.lit('Medium Order'))
        .otherwise(nw.lit('Small Order'))
        .alias('order_size'),
        
        nw.when((nw.col('category') == 'Electronics') & (nw.col('amount') > 100))
        .then(nw.col('amount') * 0.95)  # 5% loyalty discount
        .otherwise(nw.col('amount'))
        .alias('loyalty_price')
    ])
    
    print(conditional.select(['order_id', 'category', 'amount', 'order_size', 'loyalty_price']).to_pandas())
    
    # Ranking and window-like operations
    print(f"\nRanking and analytical operations:")
    
    # Group-based operations
    customer_stats = df.group_by('customer').agg([
        nw.col('amount').sum().alias('total_spent'),
        nw.col('amount').mean().alias('avg_order'),
        nw.len().alias('order_count'),
        nw.col('discount').mean().alias('avg_discount')
    ]).with_columns([
        (nw.col('total_spent') / nw.col('order_count')).alias('spend_per_order_check')
    ])
    
    print(f"Customer analytics:")
    print(customer_stats.to_pandas())
    
    # Category analysis
    category_analysis = df.group_by('category').agg([
        nw.col('amount').sum().alias('category_revenue'),
        nw.col('quantity').sum().alias('total_quantity'),
        nw.col('discount').mean().alias('avg_discount_rate'),
        nw.len().alias('order_count')
    ]).with_columns([
        (nw.col('category_revenue') / nw.col('total_quantity')).alias('revenue_per_unit')
    ])
    
    print(f"\nCategory analysis:")
    print(category_analysis.to_pandas())
    
    # Complex aggregations with filtering
    print(f"\nComplex aggregations:")
    
    high_value_summary = (
        df.filter(nw.col('amount') > 100)
        .group_by('category')
        .agg([
            nw.col('amount').min().alias('min_high_value'),
            nw.col('amount').max().alias('max_high_value'),
            (nw.col('amount') * (1 - nw.col('discount'))).sum().alias('total_final_amount'),
            nw.len().alias('high_value_count')
        ])
    )
    
    print(f"High-value orders (>$100) by category:")
    print(high_value_summary.to_pandas())
    
    # Chained transformations
    print(f"\nChained transformations:")
    
    pipeline_result = (
        df
        .with_columns([
            (nw.col('amount') * (1 - nw.col('discount'))).alias('net_amount')
        ])
        .filter(nw.col('net_amount') > 80)
        .with_columns([
            (nw.col('net_amount') / nw.col('quantity')).alias('net_price_per_unit')
        ])
        .group_by('customer')
        .agg([
            nw.col('net_amount').sum().alias('total_net_spent'),
            nw.col('net_price_per_unit').mean().alias('avg_net_price_per_unit'),
            nw.len().alias('qualified_orders')
        ])
        .filter(nw.col('total_net_spent') > 150)
        .sort('total_net_spent', descending=True)
    )
    
    print(f"Pipeline result (qualified high-value customers):")
    print(pipeline_result.to_pandas())
    
    # Show expression capabilities
    print(f"\nExpression system capabilities:")
    expression_features = [
        "Column expressions: Rich expression language for data transformation",
        "String operations: Comprehensive string manipulation functions",
        "Mathematical functions: Full range of mathematical operations",
        "Conditional logic: Complex if-then-else and case statement support",
        "Aggregation expressions: Flexible aggregation with custom expressions",
        "Chained operations: Pipeline multiple transformations efficiently"
    ]
    
    for feature in expression_features:
        print(f"  • {feature}")
    
    print("Advanced expressions and transformations completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating expression system concepts:")
    concepts = [
        "Expression language: Powerful syntax for data transformations",
        "Column operations: Mathematical, string, and logical operations",
        "Conditional logic: Complex if-then-else statements",
        "Aggregation expressions: Custom aggregations with expressions",
        "Pipeline transformations: Chain multiple operations efficiently"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise

## Example 5: Production patterns and library integration

In [ ]:
# Example 5: Production patterns and library integration
import narwhals as nw
import pandas as pd

print("Example 5: Production Patterns and Library Integration")

try:
    # Production function patterns
    print("Production function patterns for DataFrame-agnostic code:")
    
    def analyze_sales_data(df_native):
        """
        Production function that works with any supported DataFrame backend
        """
        # Convert to narwhals DataFrame
        df = nw.from_native(df_native)
        
        # Perform analysis using narwhals API
        result = (
            df
            .with_columns([
                (nw.col('revenue') * (1 - nw.col('discount_rate'))).alias('net_revenue'),
                nw.col('date').dt.month().alias('month')
            ])
            .group_by(['region', 'month'])
            .agg([
                nw.col('net_revenue').sum().alias('total_net_revenue'),
                nw.col('revenue').mean().alias('avg_revenue'),
                nw.len().alias('transaction_count')
            ])
            .filter(nw.col('total_net_revenue') > 1000)
            .sort('total_net_revenue', descending=True)
        )
        
        # Return in original backend format
        return result.to_native()
    
    # Test data
    sales_data = {
        'date': pd.date_range('2024-01-01', periods=100, freq='D'),
        'region': (['North', 'South', 'East', 'West'] * 25),
        'revenue': list(range(100, 200)),  # 100 values for 100 dates
        'discount_rate': ([0.0, 0.1, 0.05, 0.15] * 25)
    }
    
    # Create pandas DataFrame
    df_pandas = pd.DataFrame(sales_data)
    print(f"\nInput data: {df_pandas.shape[0]} records")
    print(f"Backend: pandas")
    
    # Analyze using production function
    result_pandas = analyze_sales_data(df_pandas)
    print(f"\nAnalysis result:")
    print(f"Result type: {type(result_pandas)}")
    print(f"Result shape: {result_pandas.shape}")
    print(result_pandas.head())
    
    # Library integration patterns
    print(f"\nLibrary integration patterns:")
    
    @nw.narwhalify
    def compute_metrics(df):
        """
        Narwhalify decorator automatically handles DataFrame conversion
        """
        return (
            df
            .with_columns([
                (nw.col('value') - nw.col('value').mean()).alias('deviation'),
                nw.col('value').rank().alias('rank')
            ])
            .select(['id', 'value', 'deviation', 'rank'])
        )
    
    # Test narwhalify decorator
    test_data = pd.DataFrame({
        'id': [1, 2, 3, 4, 5],
        'value': [10, 25, 15, 30, 20]
    })
    
    decorated_result = compute_metrics(test_data)
    print(f"\nNarwhalify decorator result:")
    print(decorated_result)
    
    # Error handling patterns
    print(f"\nError handling and validation patterns:")
    
    def safe_dataframe_operation(df_native, required_columns):
        """
        Production function with proper error handling
        """
        try:
            # Convert to narwhals
            df = nw.from_native(df_native)
            
            # Validate required columns
            available_columns = set(df.columns)
            missing_columns = set(required_columns) - available_columns
            
            if missing_columns:
                raise ValueError(f"Missing required columns: {missing_columns}")
            
            # Perform safe operations
            result = (
                df
                .select(required_columns)
                # narwhals 2.x: no `Expr.is_not_null`, and `all_horizontal` REQUIRES the keyword-only `ignore_nulls`
                .filter(nw.all_horizontal([~nw.col(col).is_null() for col in required_columns], ignore_nulls=False))
            )
            
            print(f"   ✓ Successfully processed {result.shape[0]} valid records")
            return result.to_native()
            
        except Exception as e:
            print(f"   ✗ Error in DataFrame operation: {e}")
            return None
    
    # Test error handling
    test_result = safe_dataframe_operation(df_pandas, ['date', 'region', 'revenue'])
    print(f"Safe operation completed: {test_result is not None}")
    assert test_result is not None and len(test_result) == 100, test_result
    
    # Performance optimization patterns
    print(f"\nPerformance optimization patterns:")
    
    def optimized_processing(df_native, use_lazy=True):
        """
        Performance-optimized DataFrame processing
        """
        df = nw.from_native(df_native)
        
        if use_lazy and hasattr(df, 'lazy'):
            # Use lazy evaluation for better performance
            result = (
                df.lazy()
                .filter(nw.col('revenue') > 150)  # Early filtering
                .select(['region', 'revenue', 'discount_rate'])  # Column pruning
                .with_columns([
                    (nw.col('revenue') * (1 - nw.col('discount_rate'))).alias('net_revenue')
                ])
                .group_by('region')
                .agg([
                    nw.col('net_revenue').sum().alias('total_net'),
                    nw.col('net_revenue').mean().alias('avg_net')
                ])
                .collect()
            )
            print("   ✓ Used lazy evaluation")
        else:
            # Eager evaluation
            result = (
                df
                .filter(nw.col('revenue') > 150)
                .select(['region', 'revenue', 'discount_rate'])
                .with_columns([
                    (nw.col('revenue') * (1 - nw.col('discount_rate'))).alias('net_revenue')
                ])
                .group_by('region')
                .agg([
                    nw.col('net_revenue').sum().alias('total_net'),
                    nw.col('net_revenue').mean().alias('avg_net')
                ])
            )
            print("   ✓ Used eager evaluation")
        
        return result.to_native()
    
    # Test performance patterns
    perf_result = optimized_processing(df_pandas, use_lazy=True)
    print(f"Performance optimization result shape: {perf_result.shape}")
    
    # Configuration and settings
    print(f"\nConfiguration patterns:")
    
    def configurable_analysis(df_native, config):
        """
        Configurable analysis function
        """
        df = nw.from_native(df_native)
        
        # Apply filters based on config
        if config.get('min_revenue'):
            df = df.filter(nw.col('revenue') >= config['min_revenue'])
        
        # Select columns based on config
        columns = config.get('columns', df.columns)
        df = df.select(columns)
        
        # Apply aggregations based on config
        if config.get('group_by'):
            agg_cols = config.get('aggregations', [nw.len().alias('count')])
            df = df.group_by(config['group_by']).agg(agg_cols)
        
        # Apply sorting based on config
        if config.get('sort_by'):
            df = df.sort(config['sort_by'], descending=config.get('descending', False))
        
        return df.to_native()
    
    # Test configuration
    analysis_config = {
        'min_revenue': 175,
        'columns': ['region', 'revenue'],
        'group_by': ['region'],
        'aggregations': [
            nw.col('revenue').sum().alias('total_revenue'),
            nw.col('revenue').mean().alias('avg_revenue')
        ],
        'sort_by': 'total_revenue',
        'descending': True
    }
    
    config_result = configurable_analysis(df_pandas, analysis_config)
    print(f"Configurable analysis result:")
    print(config_result)
    
    # Show production benefits
    print(f"\nProduction deployment benefits:")
    production_benefits = [
        "Backend independence: Write once, deploy on any DataFrame library",
        "Performance optimization: Leverage lazy evaluation where available", 
        "Error handling: Robust validation and exception management",
        "Library integration: Seamless integration with existing codebases",
        "Configuration management: Flexible, configurable data processing",
        "Migration support: Easy transition between DataFrame backends"
    ]
    
    for benefit in production_benefits:
        print(f"  • {benefit}")
    
    # Integration patterns
    print(f"\nIntegration patterns:")
    integration_patterns = [
        "Narwhalify decorator: Automatic DataFrame conversion for functions",
        "Error boundaries: Safe operations with graceful error handling",
        "Configuration-driven: Parameterized analysis functions",
        "Performance-aware: Optimize based on backend capabilities",
        "Testing-friendly: Easy to test with multiple DataFrame backends"
    ]
    
    for pattern in integration_patterns:
        print(f"  • {pattern}")
    
    print("Production patterns and integration completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating production integration concepts:")
    concepts = [
        "Production functions: Backend-agnostic data processing functions",
        "Library integration: Seamless integration with existing systems",
        "Error handling: Robust validation and exception management", 
        "Performance optimization: Leverage backend-specific capabilities",
        "Configuration patterns: Flexible, parameterized analysis workflows"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise